In [85]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.tools import WikipediaQueryRun
from langchain_tavily import TavilySearch
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import Tool

from dataclasses import dataclass
from pydantic import BaseModel
from typing import List
from langchain_core.documents import Document

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from IPython.display import display, Image
load_dotenv()


True

In [86]:
class ProcessLoader:

    def __init__(self):
        self.client = QdrantClient(
                            url=os.getenv('QDRANT_API_URL'),
                            api_key=os.getenv('QDRANT_API_KEY')
                        )
        self.dense_embedding = HuggingFaceEmbeddings(model = 'sentence-transformers/all-MiniLM-L12-v2')
        self.sparse_embedding = FastEmbedSparse(model_name ='Qdrant/bm25')
        
        self.qdrantdb = QdrantVectorStore(
            client=self.client,
            collection_name= os.getenv('COLLECTIONNAME'),
            embedding=self.dense_embedding,
            sparse_embedding=self.sparse_embedding
        )
        self.wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(top_k_results= 5, doc_content_chars_max= 2000))
        self.tavily_tool = TavilySearch(
            max_results=5,
            topic="general",
        )

    @property
    def getPBIRetriver(self):

        return self.qdrantdb.as_retriever(
            search_type = 'mmr',
                search_kwarg = {
                    "k":10
                }
        )
    
    


In [87]:
llm = ChatGroq(model='openai/gpt-oss-120b')
pbi_obj = ProcessLoader()
pbi_retriver = pbi_obj.getPBIRetriver


In [88]:
# @dataclass
class SelfReflection(BaseModel):

    question: str
    retrivedDocs : List[Document] = []
    answer: str
    reflection : str
    revised : bool
    attempts: int = 0

In [89]:
def generateRetriverDocs(state: SelfReflection) -> SelfReflection:

    """ 
    Retrieves relevant documents from a vector store based on a query. 
    This node handles the critical step of fetching context based on question attribute of state.

    :param state: object of ChainOfCmdBot include below members.
                    a. question (String)
                    b. retrivedDocs (List of Document)
                    c. answer (string)
                    d. reflection (String)
                    e. revised (string)
                    f. attempts (int)

    :type : SelfReflection

    :returs : Object of SelfReflection type where retrivedDocs will be populated by retriver (pbi_retriver)  
    """

    results = pbi_retriver.invoke(state.question)
    return state.model_copy(update={
                            "retrivedDocs" : results
                            })




In [90]:
def generateAnswer(state: SelfReflection) -> SelfReflection:

    """
        Synthesizes the final answer by consolidating retrieved documents.

        This terminal node in the LLM chain takes the list of relevant documents, 
        feeds them into the Language Model (LLM) as context, and prompts the LLM to generate a single, coherent, and
        comprehensive answer that addresses all parts of the original query.

        :param state: object of ChainOfCmdBot include below members.
                    a. question (String)
                    b. retrivedDocs (List of Document)
                    c. answer (string)
                    d. reflection (String)
                    e. revised (bool)
                    f. attempts (int)

        :type : SelfReflection

        :returs : Object of SelfReflection type where answer will be populated by LLM based on Documents received from retriver (pbi_retriver)  
    """

    context_documents = "\n\n".join([doc.page_content for doc in state.retrivedDocs])
    prompt = f""" 
             You are an expert Q&A system. Your task is to generate a single, comprehensive, and well-structured answer to the user's original query.

                    **Instructions:**
                    1.  Read the **Original Query** 
                    2.  Carefully analyze the content of the **Context Documents** provided below.
                    3.  Synthesize the information from the documents to construct a complete answer that addresses all parts of the Original Query.
                    4.  Do not introduce any information that is not explicitly present in the Context Documents.
                    5.  Structure your final answer clearly using markdown headings and bullet points where appropriate.
                    6.  Refer Docstring for more details

                    **Original Query:**
                    {state.question}

                    **Context Documents:**
                    {context_documents}

                    **Final Answer:**

            """
    
    result = llm.invoke(prompt).content
    return state.model_copy(
        update={
            "answer": result,
            "attempts" : state.attempts + 1
        }
    )

In [91]:
def reflectionEval(state: SelfReflection) -> SelfReflection:

    """ 
    Evaluates and critiques the generated answer against provided context and query.

    This node performs the crucial self-reflection step in the RAG or Agentic workflow.
    It invokes an LLM to act as a 'Critic' or 'Judge', assessing the 'initial_answer'
    for factual grounding (against retrieved_context), completeness (against original_query),
    and adherence to format.

    The primary output is a binary decision or a critique, which is used by the
    LangGraph router to determine the workflow's next step.

    :param state: object of ChainOfCmdBot include below members.
                    a. question (String)
                    b. retrivedDocs (List of Document)
                    c. answer (string)
                    d. reflection (String)
                    e. revised (bool)
                    f. attempts (int)

    :type : SelfReflection    

    :returns ::returs : Object of SelfReflection type where answer will be evaluated by LLM and populate reflection and revised attributes.
"""
    context_documents = '\n\n'.join([doc.page_content for doc in state.retrivedDocs])
    prompt = f""" 
            **System Instruction: Reflection and Critique**

                Your role is to act as a rigorous, unbiased editor and quality assurance critic for the generated **Initial Answer**. 
                You must evaluate the provided answer against the original **Query** and the **Context** based on the criteria below.

                **Instructions:**
                    1.  **Analyze** the Initial Answer based on the **Evaluation Criteria**.
                    2.  If the Initial Answer is **perfect**, output only the single word: **YES**.
                    3.  If the Initial Answer **fails** any criteria, generate a detailed critique.
                    4.  IF LLM found Question is not relevant to context then Mark Initial Answer as Fail.
                    

                **Evaluation Criteria:**
                    * **Completeness:** Does the Initial Answer address **every component** of the Original Query? Are all sub-questions answered?
                    * **Grounding:** Is **all factual information** strictly supported by the **Retrieved Context**? Are any statements speculative or hallucinated?
                    * **Clarity and Coherence:** Is the answer logically structured, easy to read, and free of contradictions or confusing language?
                    * **Format Compliance:** Does the answer adhere to the required output constraints (e.g., specific markdown headings, JSON format, length limits)?

            **Inputs to Evaluate:**

            **Original Query:**
            {state.question}

            **Retrieved Context (The only source of truth):**
            {context_documents}

            **Initial Answer (The output you must critique):**
            {state.answer}

            ---

            **CRITIQUE & REFINEMENT**

            **Critique:** (Provide a detailed, point-by-point analysis based on the Evaluation Criteria. 
                Specify exactly *what* is wrong and *where* in the context the correct information can be found.)

            **Refined Action/Correction:** (Provide precise, actionable steps or corrections required to fix the Initial Answer. 
                If the answer is [FAIL], rewrite the problematic section or clearly indicate the missing information.)
        """
    
    result = llm.invoke(prompt).content
    return state.model_copy(update= {
                                        "reflection": result ,
                                        reversed: 1 if result.lower() == "yes" else 0
                                    } 
                            )
    

In [93]:
obj = SelfReflection(
 question = "What is the Difference Power BI and ML",
    retrivedDocs = [],
    answer = "",
    reflection = "",
    revised = "YES",
    attempts = 0   
)

obj = generateRetriverDocs(obj)
obj = generateAnswer(obj)
print('-----------------------------------------------------')
obj = reflectionEval(obj)
print(obj.question)
print('-----------------------------------------------------')
print(obj.answer)
print('-----------------------------------------------------')
print(obj.reflection, obj.revised)
print('-----------------------------------------------------')

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}